# URL EDA and model comparisons
Run from the repository root with its installed environment. Training logic lives in src, never here. No URL is fetched. Real data must be downloaded manually first. Synthetic test metrics are not a research result.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px

from phishlens.features.url_offline import extract_features
from phishlens.models.data import load_urls

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
data = load_urls(root / "data/urls.csv")
data.groupby("label").agg(rows=("url", "size"), domains=("domain", "nunique"))

In [ ]:
features = pd.DataFrame(data.url.map(extract_features).tolist())
features.describe()

In [ ]:
eda = features.assign(label=data.label.map({0: "legitimate", 1: "phishing"}))
px.histogram(eda, x="url_length", color="label", barmode="overlay", log_y=True).show()
px.box(eda, x="label", y="host_entropy").show()

## Train outside the notebook
Run `python -m phishlens.models.train_url --data data/urls.csv` from the repository root. Inspect source provenance and class balance before interpreting metrics. Domain grouping reduces memorization leakage, but does not remove source bias.

In [ ]:
metadata = json.loads((root / "artifacts/url_offline.metadata.json").read_text())
comparison = pd.DataFrame(metadata["validation_metrics"]).T
px.bar(
    comparison.reset_index(), x="index", y=["precision", "recall", "f1", "roc_auc"], barmode="group"
).show()
print(metadata["selection_reason"])
print("Held-out test:", metadata["test_metrics"])

In [ ]:
matrix = metadata["test_metrics"]["confusion_matrix"]
px.imshow(
    matrix,
    x=["legitimate", "phishing"],
    y=["legitimate", "phishing"],
    text_auto=True,
    labels={"x": "Predicted", "y": "Actual"},
).show()